# Revised MJR8396C manuscript simulations and figures

This notebook regenerates manuscript-facing results using the final gap-filled **Megasphaera sp. MJR8396C** GEM and three controlled derivatives of the revised anaerobic medium:

1. **Gut-like:** the complete gap-filled anaerobic base medium, interpreted in relation to the EU-average AGORA diet.
2. **Carbon-rich:** the gut-like medium with the overlapping carbohydrate D-glucose scaled by the carbon-doubled/EU-average AGORA ratio.
3. **AA-rich:** the gut-like medium with the six overlapping amino acids scaled by High-protein/EU-average AGORA ratios.

The notebook exports reproducible CSV/TSV tables and separate publication-quality PNG/SVG/PDF figures. It does not overwrite the original project outputs.

## Recommended manuscript use

- **Revised Figure 1A:** reconstruction/model-quality workflow (schematic prepared in the manuscript, using the exported model statistics).
- **Revised Figure 1B:** final GEM statistics and MEMOTE summary.
- **Revised Figure 1C:** heatmap of relative nutrient bounds in the three media.
- **Revised Figure 1D:** predicted growth across the three media.
- **Revised Table 2:** growth, uptake, and fermentation outputs for gut-like, carbon-rich, and AA-rich media.
- **Revised Figure 2:** carbon-normalized substrate screen and amino-acid-pair analysis.
- **Revised Figure 3:** MICOM tradeoff and cross-feeding figures from the existing project outputs.
- **Revised Figure 4 or Supplement:** butyrate production envelope and constraint-based strain-design outputs.

In [ ]:
from pathlib import Path
import itertools
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import cobra
    from cobra.io import read_sbml_model
    from cobra.flux_analysis import pfba, flux_variability_analysis
except ImportError as exc:
    raise ImportError(
        "COBRApy is required. Install with: pip install cobra swiglpk pandas numpy matplotlib"
    ) from exc

# Edit only PROJECT if the package is placed elsewhere.
PROJECT = Path('/Users/wtscott/DataspellProjects/Magesphaera_Project').resolve()
MODEL_DIR = PROJECT / 'inputs' / 'models'
MEDIA_DIR = PROJECT / 'inputs' / 'media'
OUT = PROJECT / 'outputs' / 'revised_manuscript'
FIG = OUT / 'figures'
TAB = OUT / 'tables'
for directory in (OUT, FIG, TAB):
    directory.mkdir(parents=True, exist_ok=True)

MODEL_FILE = MODEL_DIR / 'Megasphaera_sp_MJR8396C_filled_anaerobic.xml'
MEDIA_FILES = {
    'Gut-like': MEDIA_DIR / 'MJR8396C_gut_like_medium.tsv',
    'Carbon-rich': MEDIA_DIR / 'MJR8396C_carbon_rich_medium.tsv',
    'AA-rich': MEDIA_DIR / 'MJR8396C_AA_rich_medium.tsv',
}

assert MODEL_FILE.exists(), MODEL_FILE
for name, path in MEDIA_FILES.items():
    assert path.exists(), f'{name}: {path}'

model = read_sbml_model(str(MODEL_FILE))
print(model)
print('Solver:', model.solver.interface.__name__)

## 1. Helpers: model, exchange reactions, media, and exports

In [ ]:
KEY_CPDS = {
    'cpd00027': 'D-glucose',
    'cpd00159': 'L-lactate',
    'cpd00221': 'D-lactate',
    'cpd00029': 'acetate',
    'cpd00211': 'butyrate',
    'cpd00141': 'propionate',
    'cpd00047': 'formate',
    'cpd00036': 'succinate',
    'cpd00013': 'ammonia',
    'cpd00239': 'hydrogen sulfide',
    'cpd00324': 'methanethiol',
    'cpd00011': 'carbon dioxide',
}

BIOMASS_CANDIDATES = ['bio1', 'BIOMASS', 'biomass']

def find_biomass(model):
    for rid in BIOMASS_CANDIDATES:
        if rid in model.reactions:
            return rid
    objectives = [r.id for r in model.reactions if abs(r.objective_coefficient) > 0]
    if objectives:
        return objectives[0]
    candidates = [r.id for r in model.reactions if 'biomass' in r.id.lower() or 'biomass' in (r.name or '').lower()]
    if not candidates:
        raise ValueError('No biomass reaction found.')
    return candidates[0]

BIOMASS = find_biomass(model)
print('Biomass reaction:', BIOMASS)

def cpd_from_reaction(rid):
    hit = re.search(r'(cpd\d+)', str(rid))
    return hit.group(1) if hit else None

def exchange_map(model):
    rows=[]
    for reaction in model.exchanges:
        cpd = cpd_from_reaction(reaction.id)
        rows.append({
            'reaction': reaction.id,
            'reaction_name': reaction.name,
            'compound': cpd,
            'compound_name': KEY_CPDS.get(cpd, ''),
            'lower_bound': reaction.lower_bound,
            'upper_bound': reaction.upper_bound,
        })
    return pd.DataFrame(rows)

EXCHANGE_TABLE = exchange_map(model)
EXCHANGE_TABLE.to_csv(TAB / 'final_model_exchange_reactions.csv', index=False)

def find_exchange(model, cpd):
    candidates = [r for r in model.exchanges if cpd in r.id]
    if not candidates:
        candidates = [r for r in model.reactions if r.id.startswith('EX_') and cpd in r.id]
    if not candidates:
        return None
    # Prefer extracellular e0 exchange over shared-medium reactions.
    candidates = sorted(candidates, key=lambda r: (not r.id.endswith('_e0'), len(r.id)))
    return candidates[0].id

EX_BY_CPD = {cpd: find_exchange(model, cpd) for cpd in set(KEY_CPDS)}
print(pd.Series(EX_BY_CPD, name='reaction'))

def read_medium(path):
    df = pd.read_csv(path, sep='\t')
    required = {'compounds','name','maxFlux'}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f'{path} missing columns: {sorted(missing)}')
    df['maxFlux'] = pd.to_numeric(df['maxFlux'], errors='raise')
    return df

def apply_medium(model, medium_df, close_all_uptake=True):
    m = model.copy()
    if close_all_uptake:
        for reaction in m.exchanges:
            reaction.lower_bound = max(0.0, reaction.lower_bound)
    mapping=[]
    for row in medium_df.itertuples(index=False):
        rid = find_exchange(m, row.compounds)
        if rid is None:
            mapping.append({'compound': row.compounds, 'name': row.name, 'reaction': None, 'maxFlux': row.maxFlux, 'mapped': False})
            continue
        r = m.reactions.get_by_id(rid)
        r.lower_bound = -float(row.maxFlux)
        # Allow secretion unless reaction is explicitly irreversible.
        r.upper_bound = max(float(r.upper_bound), 1000.0)
        mapping.append({'compound': row.compounds, 'name': row.name, 'reaction': rid, 'maxFlux': row.maxFlux, 'mapped': True})
    return m, pd.DataFrame(mapping)

def savefig_all(fig, stem, dpi=600):
    fig.tight_layout()
    for ext in ('png','svg','pdf'):
        fig.savefig(FIG / f'{stem}.{ext}', dpi=dpi if ext == 'png' else None, bbox_inches='tight')
    plt.show()
    plt.close(fig)

## 2. Final model statistics and medium mapping

In [ ]:
model_stats = pd.DataFrame([{
    'model_id': model.id,
    'biomass_reaction': BIOMASS,
    'genes': len(model.genes),
    'reactions': len(model.reactions),
    'metabolites': len(model.metabolites),
    'exchange_reactions': len(model.exchanges),
    'transport_reactions': sum(len({m.compartment for m in r.metabolites}) > 1 for r in model.reactions),
}])
model_stats.to_csv(TAB / 'revised_model_statistics.csv', index=False)
display(model_stats)

mapping_rows=[]
media_data={}
for medium_name, path in MEDIA_FILES.items():
    df = read_medium(path)
    media_data[medium_name] = df
    _, mp = apply_medium(model, df)
    mp.insert(0, 'medium', medium_name)
    mapping_rows.append(mp)
medium_mapping = pd.concat(mapping_rows, ignore_index=True)
medium_mapping.to_csv(TAB / 'revised_media_exchange_mapping.csv', index=False)
display(medium_mapping.groupby('medium')['mapped'].agg(['sum','count']))

## 3. Revised Figure 1C: medium-bound comparison

In [ ]:
# Compare only compounds whose bounds vary, plus biologically central substrates.
all_media = []
for medium_name, df in media_data.items():
    temp = df[['compounds','name','maxFlux']].copy()
    temp['medium'] = medium_name
    all_media.append(temp)
medium_long = pd.concat(all_media, ignore_index=True)
medium_wide = medium_long.pivot_table(index=['compounds','name'], columns='medium', values='maxFlux', aggfunc='first').fillna(0)
variable = medium_wide.nunique(axis=1) > 1
central_cpds = {'cpd00027','cpd00159','cpd00029','cpd00041','cpd00033','cpd00039','cpd00060','cpd00084','cpd00119'}
keep = variable | medium_wide.index.get_level_values('compounds').isin(central_cpds)
plot_df = medium_wide.loc[keep].copy()
plot_df = plot_df[['Gut-like','Carbon-rich','AA-rich']]
plot_df.index = [name for _, name in plot_df.index]

# Display relative to gut-like to prevent 1000-unit inorganic bounds dominating.
relative = plot_df.div(plot_df['Gut-like'].replace(0, np.nan), axis=0).fillna(0)
fig, ax = plt.subplots(figsize=(7.5, max(4.5, 0.36*len(relative))))
im = ax.imshow(relative.values, aspect='auto')
ax.set_xticks(range(len(relative.columns)), relative.columns, rotation=25, ha='right')
ax.set_yticks(range(len(relative.index)), relative.index)
for i in range(relative.shape[0]):
    for j in range(relative.shape[1]):
        value = relative.iloc[i,j]
        ax.text(j, i, f'{value:.2f}×', ha='center', va='center', fontsize=8)
ax.set_title('Relative nutrient uptake bounds across revised media')
fig.colorbar(im, ax=ax, label='Bound relative to gut-like medium')
savefig_all(fig, 'Figure_1C_revised_medium_bound_heatmap')

medium_wide.reset_index().to_csv(TAB / 'revised_media_bound_comparison.csv', index=False)

## 4. Three-medium pFBA and FVA analysis: revised Table 2 and Figure 1D

In [ ]:
TARGET_CPDS = ['cpd00027','cpd00159','cpd00221','cpd00029','cpd00211','cpd00141','cpd00047','cpd00036','cpd00013','cpd00239','cpd00324','cpd00011']

def run_medium_scenario(base_model, medium_name, medium_df, fva_fraction=0.90):
    m, mapping = apply_medium(base_model, medium_df)
    m.objective = BIOMASS
    optimum = m.optimize()
    if optimum.status != 'optimal':
        raise RuntimeError(f'{medium_name}: FBA status={optimum.status}')
    psol = pfba(m)
    row = {
        'medium': medium_name,
        'growth_rate': float(optimum.objective_value),
        'pfba_biomass_flux': float(psol.fluxes.get(BIOMASS, np.nan)),
        'mapped_medium_compounds': int(mapping['mapped'].sum()),
        'total_medium_compounds': int(len(mapping)),
    }
    target_rxns=[]
    for cpd in TARGET_CPDS:
        rid = find_exchange(m, cpd)
        name = KEY_CPDS.get(cpd, cpd)
        row[f'{name}_reaction'] = rid
        row[f'{name}_flux'] = float(psol.fluxes.get(rid, np.nan)) if rid else np.nan
        if rid:
            target_rxns.append(rid)
    # FVA at 90% of optimum for biomass and target exchanges.
    fva_rxns = list(dict.fromkeys([BIOMASS] + target_rxns))
    fva = flux_variability_analysis(m, reaction_list=fva_rxns, fraction_of_optimum=fva_fraction)
    fva = fva.reset_index().rename(columns={'index':'reaction'})
    fva.insert(0, 'medium', medium_name)
    return row, fva, psol

scenario_rows=[]
fva_rows=[]
solutions={}
for medium_name, df in media_data.items():
    row, fva, sol = run_medium_scenario(model, medium_name, df)
    scenario_rows.append(row)
    fva_rows.append(fva)
    solutions[medium_name] = sol

scenario_summary = pd.DataFrame(scenario_rows)
fva_summary = pd.concat(fva_rows, ignore_index=True)
scenario_summary.to_csv(TAB / 'Table_2_revised_three_medium_simulation_summary.csv', index=False)
fva_summary.to_csv(TAB / 'Table_S_revised_three_medium_FVA_90pct.csv', index=False)
display(scenario_summary)

In [ ]:
# Revised Figure 1D: growth across media.
fig, ax = plt.subplots(figsize=(6.8,4.5))
ax.bar(scenario_summary['medium'], scenario_summary['growth_rate'])
ax.set_ylabel('Maximum biomass flux')
ax.set_title('Predicted MJR8396C growth across revised media')
ax.tick_params(axis='x', rotation=20)
for i, value in enumerate(scenario_summary['growth_rate']):
    ax.text(i, value, f'{value:.3f}', ha='center', va='bottom')
savefig_all(fig, 'Figure_1D_revised_growth_across_media')

In [ ]:
# Separate manuscript-ready fermentation-product heatmap.
product_names = ['butyrate','propionate','acetate','formate','succinate','ammonia','hydrogen sulfide','methanethiol','L-lactate','D-lactate']
product_cols = [f'{name}_flux' for name in product_names if f'{name}_flux' in scenario_summary.columns]
product_flux = scenario_summary.set_index('medium')[product_cols].copy()
product_flux.columns = [c.removesuffix('_flux') for c in product_flux.columns]
fig, ax = plt.subplots(figsize=(9.5,4.2))
im = ax.imshow(product_flux.values, aspect='auto')
ax.set_xticks(range(len(product_flux.columns)), product_flux.columns, rotation=35, ha='right')
ax.set_yticks(range(len(product_flux.index)), product_flux.index)
for i in range(product_flux.shape[0]):
    for j in range(product_flux.shape[1]):
        ax.text(j, i, f'{product_flux.iloc[i,j]:.2f}', ha='center', va='center', fontsize=8)
ax.set_title('pFBA exchange fluxes across revised media')
fig.colorbar(im, ax=ax, label='Exchange flux (positive secretion; negative uptake)')
savefig_all(fig, 'Figure_S_revised_three_medium_product_flux_heatmap')
product_flux.to_csv(TAB / 'revised_three_medium_key_product_fluxes.csv')

## 5. Carbon-normalized substrate screen

The screen fixes carbon-atom influx rather than assigning an equal molar uptake to every substrate. For a substrate with \(C_i\) carbon atoms and target carbon influx \(C^*\), the uptake bound is \(u_i=C^*/C_i\). This avoids favoring compounds solely because they carry more carbon per molecule.

In [ ]:
# Candidate carbon substrates. Extend only when the final model has a mapped exchange.
CARBON_SOURCES = {
    'D-glucose': ('cpd00027', 6),
    'L-lactate': ('cpd00159', 3),
    'D-lactate': ('cpd00221', 3),
    'acetate': ('cpd00029', 2),
    'pyruvate': ('cpd00020', 3),
    'D-ribose': ('cpd00105', 5),
    'D-xylose': ('cpd00154', 5),
    'D-fructose': ('cpd00082', 6),
    'sucrose': ('cpd00076', 12),
    'maltose': ('cpd00179', 12),
}
TARGET_CARBON_INFLUX = 60.0

# Keep inorganic/vitamin background, close organic carbon in the base medium, then add one source.
organic_base_cpds = set(CARBON_SOURCES[name][0] for name in CARBON_SOURCES) | {'cpd00159','cpd00029'}
background = media_data['Gut-like'].copy()
background.loc[background['compounds'].isin(organic_base_cpds), 'maxFlux'] = 0.0

carbon_rows=[]
for substrate, (cpd, carbon_atoms) in CARBON_SOURCES.items():
    rid = find_exchange(model, cpd)
    if rid is None:
        carbon_rows.append({'substrate':substrate,'compound':cpd,'mapped':False})
        continue
    test_medium = background.copy()
    bound = TARGET_CARBON_INFLUX / carbon_atoms
    if cpd in set(test_medium['compounds']):
        test_medium.loc[test_medium['compounds'].eq(cpd), 'maxFlux'] = bound
    else:
        test_medium = pd.concat([test_medium, pd.DataFrame([{'compounds':cpd,'name':substrate,'maxFlux':bound}])], ignore_index=True)
    m, _ = apply_medium(model, test_medium)
    m.objective = BIOMASS
    opt = m.optimize()
    row={'substrate':substrate,'compound':cpd,'mapped':True,'carbon_atoms':carbon_atoms,'uptake_bound':bound,'carbon_influx':TARGET_CARBON_INFLUX,'status':opt.status}
    if opt.status == 'optimal':
        ps = pfba(m)
        row['growth_rate'] = float(opt.objective_value)
        row['growth_per_carbon'] = float(opt.objective_value)/TARGET_CARBON_INFLUX
        for target in ['cpd00211','cpd00141','cpd00029','cpd00047','cpd00036']:
            tr = find_exchange(m,target)
            row[f'{KEY_CPDS.get(target,target)}_flux'] = float(ps.fluxes.get(tr,np.nan)) if tr else np.nan
            row[f'{KEY_CPDS.get(target,target)}_yield_per_C'] = row[f'{KEY_CPDS.get(target,target)}_flux']/TARGET_CARBON_INFLUX if tr else np.nan
    carbon_rows.append(row)

carbon_screen = pd.DataFrame(carbon_rows)
carbon_screen.to_csv(TAB / 'revised_carbon_normalized_substrate_screen.csv', index=False)
display(carbon_screen)

In [ ]:
plot = carbon_screen.query("mapped == True and status == 'optimal'").sort_values('growth_per_carbon', ascending=False)
fig, ax = plt.subplots(figsize=(8.5,4.8))
ax.bar(plot['substrate'], plot['growth_per_carbon'])
ax.set_ylabel('Biomass flux per mmol C supplied')
ax.set_title('Carbon-normalized growth efficiency')
ax.tick_params(axis='x', rotation=35)
savefig_all(fig, 'Figure_2A_carbon_normalized_growth_efficiency')

In [ ]:
scfa_cols = [c for c in carbon_screen.columns if c.endswith('_yield_per_C')]
scfa = carbon_screen.query("mapped == True and status == 'optimal'").set_index('substrate')[scfa_cols].copy()
scfa.columns = [c.replace('_yield_per_C','') for c in scfa.columns]
fig, ax = plt.subplots(figsize=(8.8, max(4.2,0.38*len(scfa))))
im=ax.imshow(scfa.values, aspect='auto')
ax.set_xticks(range(len(scfa.columns)), scfa.columns, rotation=35, ha='right')
ax.set_yticks(range(len(scfa.index)), scfa.index)
for i in range(scfa.shape[0]):
    for j in range(scfa.shape[1]):
        ax.text(j,i,f'{scfa.iloc[i,j]:.3f}',ha='center',va='center',fontsize=8)
ax.set_title('SCFA yield per supplied carbon across substrates')
fig.colorbar(im,ax=ax,label='Flux per mmol C supplied')
savefig_all(fig, 'Figure_2B_carbon_source_SCFA_yields')

## 6. Amino-acid-pair-dependent fermentation analysis

In [ ]:
AA_SOURCES = {
    'L-aspartate':'cpd00041',
    'glycine':'cpd00033',
    'L-lysine':'cpd00039',
    'L-methionine':'cpd00060',
    'L-cysteine':'cpd00084',
    'L-histidine':'cpd00119',
}
AA_BOUND_PER_MEMBER = 2.0

# Use the gut-like medium but set the six tested amino acids to zero, then add pairs.
aa_background = media_data['Gut-like'].copy()
aa_background.loc[aa_background['compounds'].isin(AA_SOURCES.values()), 'maxFlux'] = 0.0

aa_pair_rows=[]
for (name_a, cpd_a), (name_b, cpd_b) in itertools.combinations(AA_SOURCES.items(), 2):
    test = aa_background.copy()
    for name, cpd in [(name_a,cpd_a),(name_b,cpd_b)]:
        if cpd in set(test['compounds']):
            test.loc[test['compounds'].eq(cpd),'maxFlux'] = AA_BOUND_PER_MEMBER
        else:
            test = pd.concat([test,pd.DataFrame([{'compounds':cpd,'name':name,'maxFlux':AA_BOUND_PER_MEMBER}])],ignore_index=True)
    m, mp = apply_medium(model,test)
    m.objective=BIOMASS
    opt=m.optimize()
    row={'amino_acid_1':name_a,'amino_acid_2':name_b,'pair':f'{name_a} + {name_b}','status':opt.status}
    if opt.status=='optimal':
        ps=pfba(m)
        row['growth_rate']=float(opt.objective_value)
        for target in ['cpd00211','cpd00141','cpd00029','cpd00047','cpd00036','cpd00013','cpd00239','cpd00324']:
            rid=find_exchange(m,target)
            row[f'{KEY_CPDS.get(target,target)}_flux']=float(ps.fluxes.get(rid,np.nan)) if rid else np.nan
    aa_pair_rows.append(row)

aa_pairs=pd.DataFrame(aa_pair_rows)
aa_pairs.to_csv(TAB/'revised_amino_acid_pair_simulations.csv',index=False)
display(aa_pairs.sort_values('growth_rate',ascending=False))

In [ ]:
# Symmetric growth heatmap.
aa_names=list(AA_SOURCES)
growth_matrix=pd.DataFrame(np.nan,index=aa_names,columns=aa_names)
for row in aa_pairs.itertuples(index=False):
    growth_matrix.loc[row.amino_acid_1,row.amino_acid_2]=row.growth_rate
    growth_matrix.loc[row.amino_acid_2,row.amino_acid_1]=row.growth_rate
fig,ax=plt.subplots(figsize=(7.2,6.2))
im=ax.imshow(growth_matrix.values,aspect='equal')
ax.set_xticks(range(len(aa_names)),aa_names,rotation=40,ha='right')
ax.set_yticks(range(len(aa_names)),aa_names)
for i in range(len(aa_names)):
    for j in range(len(aa_names)):
        value=growth_matrix.iloc[i,j]
        if np.isfinite(value):
            ax.text(j,i,f'{value:.3f}',ha='center',va='center',fontsize=8)
ax.set_title('Predicted growth across amino-acid pairs')
fig.colorbar(im,ax=ax,label='Maximum biomass flux')
savefig_all(fig,'Figure_2C_amino_acid_pair_growth_heatmap')
growth_matrix.to_csv(TAB/'revised_amino_acid_pair_growth_matrix.csv')

In [ ]:
# Product heatmap for the strongest-growing pairs.
top_pairs=aa_pairs.sort_values('growth_rate',ascending=False).head(12).set_index('pair')
product_cols=[c for c in top_pairs.columns if c.endswith('_flux')]
product_matrix=top_pairs[product_cols].copy()
product_matrix.columns=[c.removesuffix('_flux') for c in product_matrix.columns]
fig,ax=plt.subplots(figsize=(9.5,max(5,0.36*len(product_matrix))))
im=ax.imshow(product_matrix.values,aspect='auto')
ax.set_xticks(range(len(product_matrix.columns)),product_matrix.columns,rotation=35,ha='right')
ax.set_yticks(range(len(product_matrix.index)),product_matrix.index)
for i in range(product_matrix.shape[0]):
    for j in range(product_matrix.shape[1]):
        ax.text(j,i,f'{product_matrix.iloc[i,j]:.2f}',ha='center',va='center',fontsize=7)
ax.set_title('Fermentation products for top amino-acid pairs')
fig.colorbar(im,ax=ax,label='Exchange flux')
savefig_all(fig,'Figure_2D_amino_acid_pair_product_heatmap')

## 7. Butyrate production envelope and strain-design-ready outputs

In [ ]:
BUTYRATE_EX = find_exchange(model,'cpd00211')
if BUTYRATE_EX is None:
    raise ValueError('Butyrate exchange reaction was not found.')

gut_model,_=apply_medium(model,media_data['Gut-like'])
gut_model.objective=BIOMASS
max_growth=float(gut_model.slim_optimize())
fractions=np.linspace(0,1,21)
envelope=[]
for fraction in fractions:
    m=gut_model.copy()
    biomass=m.reactions.get_by_id(BIOMASS)
    biomass.lower_bound=fraction*max_growth
    m.objective=BUTYRATE_EX
    max_but=float(m.slim_optimize())
    m.objective={m.reactions.get_by_id(BUTYRATE_EX):-1.0}
    min_but=-float(m.slim_optimize())
    envelope.append({'growth_fraction':fraction,'required_biomass':fraction*max_growth,'min_butyrate':min_but,'max_butyrate':max_but})
envelope=pd.DataFrame(envelope)
envelope.to_csv(TAB/'revised_butyrate_production_envelope.csv',index=False)

fig,ax=plt.subplots(figsize=(7,4.8))
ax.fill_between(envelope['required_biomass'],envelope['min_butyrate'],envelope['max_butyrate'],alpha=0.25)
ax.plot(envelope['required_biomass'],envelope['max_butyrate'],label='Maximum butyrate')
ax.plot(envelope['required_biomass'],envelope['min_butyrate'],label='Minimum butyrate')
ax.set_xlabel('Required biomass flux')
ax.set_ylabel('Butyrate exchange flux')
ax.set_title('Butyrate production envelope in the revised gut-like medium')
ax.legend()
savefig_all(fig,'Figure_4A_revised_butyrate_production_envelope')

### Optional StrainDesign analysis

Run only after installing a MILP-capable solver and `straindesign`. Results from StrainDesign must not be called “OptForce” unless the OptForce algorithm itself is run. If the current manuscript retains “OptForce,” either rerun OptForce explicitly or rename the section to “constraint-based strain-design predictions.”

In [ ]:
RUN_STRAINDESIGN = False
if RUN_STRAINDESIGN:
    try:
        import straindesign as sd
    except ImportError as exc:
        raise ImportError('Install with: pip install straindesign') from exc
    # Insert the final, solver-validated intervention formulation here.
    # Keep the output supplementary unless all intervention sets are independently verified.

## 8. Integrate existing MICOM outputs into revised Figure 3

In [ ]:
# This section uses outputs already generated in Megasphaera_Project.
EXISTING_TAB = PROJECT / 'outputs' / 'tables'
EXISTING_FIG = PROJECT / 'outputs' / 'figures'

required_micom = {
    'tradeoff':'micom_tradeoff_both_members_growth_summary.csv',
    'members':'micom_primary_member_growth_fraction_0.95.csv',
    'exchange':'manuscript_micom_lactate_scfa_exchange_table_fraction_0.95.csv',
}
for label, name in required_micom.items():
    path=EXISTING_TAB/name
    if path.exists():
        print(label, 'found:', path)
    else:
        warnings.warn(f'Missing MICOM table: {path}')

# Copy/reference the already generated figures rather than rerunning MICOM here:
# micom_tradeoff_member_growth_scan_fraction_0.95.*
# micom_primary_member_growth_fraction_0.95.*
# micom_lactate_scfa_exchange_heatmap_fraction_0.95.*
# micom_directed_exchange_network_fraction_0.95.*

## 9. Manuscript synchronization checklist

In [ ]:
manifest = pd.DataFrame([
    {'manuscript_item':'Figure 1A','source':'reconstruction workflow + gapseq/MEMOTE provenance','status':'compose schematic from verified workflow'},
    {'manuscript_item':'Figure 1B','source':'revised_model_statistics.csv + MEMOTE report','status':'regenerate'},
    {'manuscript_item':'Figure 1C','source':'Figure_1C_revised_medium_bound_heatmap','status':'generated by this notebook'},
    {'manuscript_item':'Figure 1D','source':'Figure_1D_revised_growth_across_media','status':'generated by this notebook'},
    {'manuscript_item':'Table 2','source':'Table_2_revised_three_medium_simulation_summary.csv','status':'generated by this notebook'},
    {'manuscript_item':'Figure 2A-B','source':'carbon-normalized substrate screen','status':'generated by this notebook'},
    {'manuscript_item':'Figure 2C-D','source':'amino-acid-pair simulations','status':'generated by this notebook'},
    {'manuscript_item':'Figure 3','source':'existing MICOM tradeoff and exchange outputs','status':'reuse current revised outputs'},
    {'manuscript_item':'Figure 4','source':'revised butyrate envelope + optional verified strain design','status':'generate envelope; verify strain-design terminology'},
    {'manuscript_item':'Table 3','source':'model quality and literature benchmarking','status':'add or renumber current Table 4'},
])
manifest.to_csv(TAB/'manuscript_figure_table_source_manifest.csv',index=False)
display(manifest)

## Interpretation guardrails

1. Report gapseq post-gapfilling values as **feasibility/growth predictions under the gapfilling medium**, not as experimental growth rates.
2. Report the three derivative-media outputs separately as downstream controlled nutrient-regime simulations.
3. Use positive exchange flux for secretion and negative exchange flux for uptake.
4. Describe MICOM output as model-predicted, directed cross-feeding; do not call it direct experimental validation.
5. Use “lactate-centered, acetate-assisted cross-feeding” for the current MICOM result.
6. Do not call StrainDesign/OptKnock results “OptForce.”
7. Preserve the submitted manuscript structure where possible, but regenerate every quantitative panel that depended on the former GEM or media.